# Notebook 05 — Gradio UI Demo
Launches the Hurricane Event Summariser web interface inline in the notebook,
or as a standalone app.

**Requirements:**
- All API keys set in `../.env`
- All dependencies installed (`pip install -r ../requirements.txt`)

**Docs:** https://www.gradio.app/docs/gradio/blocks

In [ ]:
import sys
sys.path.insert(0, '..')

# Verify environment before launching
from config import MISTRAL_API_KEY, BRAVE_API_KEY, ELEVENLABS_API_KEY

checks = {
    'MISTRAL_API_KEY (required)':    bool(MISTRAL_API_KEY),
    'BRAVE_API_KEY (optional news)': bool(BRAVE_API_KEY),
    'ELEVENLABS_API_KEY (TTS)':      bool(ELEVENLABS_API_KEY),
}
for name, ok in checks.items():
    status = '✓' if ok else '✗ NOT SET'
    print(f'  {status}  {name}')

if not MISTRAL_API_KEY:
    print('\nWARNING: MISTRAL_API_KEY is required for AI summaries.')
    print('Set it in hurricane_tracker/.env before launching the UI.')

## Option A — Launch Inline in Notebook
The Gradio interface will appear below this cell.

In [ ]:
from app import demo

demo.launch(
    inline=True,    # Embeds the UI in notebook output
    share=False,
    show_error=True,
)

## Option B — Launch as Standalone Server
Run this cell to start the app at http://localhost:7860 in a separate process.
Then open that URL in your browser.

In [ ]:
# Uncomment and run this cell instead of Option A if you want a standalone server:
# import subprocess, sys
# proc = subprocess.Popen([sys.executable, '../app.py'])
# print(f'App started (PID {proc.pid}) — open http://localhost:7860')
# # To stop: proc.terminate()

## Quick Function-Level Demo (No UI)
Test the full pipeline programmatically without the Gradio interface.

In [ ]:
from app import run_full_analysis

# Runs the same pipeline as clicking the button in the UI.
# run_full_analysis returns a 6-tuple:
#   (threat_badge_html, status_md, summary_text, audio_path, map_html, btn_update)
badge_html, status_md, summary_text, audio_path, map_html, _ = run_full_analysis(
    location_input='Tampa, FL',
    enable_tts=False,   # Set True if ELEVENLABS_API_KEY is configured
    historical_mode=False,
    historical_dt_str='',
)

print('=== THREAT BADGE ===')
print(badge_html[:100], '...')
print()
print('=== STATUS MARKDOWN ===')
print(status_md)
print()
print('=== AI SUMMARY ===')
print(summary_text)

if audio_path:
    print(f'\nAudio saved: {audio_path}')


## Background Monitoring Demo

In [ ]:
# Demonstrates starting and stopping the background scheduler
from geocoder import geocode_user_location
from scheduler import start_scheduler, stop_scheduler
import time

geo   = geocode_user_location('Miami, FL')
sched = start_scheduler(
    lat=geo['lat'],
    lon=geo['lon'],
    interval_minutes=1,
)
print('Scheduler running — will poll NHC every 1 minute.')
print('Waiting 5 seconds then stopping...')
time.sleep(5)
stop_scheduler(sched)
print('Scheduler stopped.')
